In [ ]:
# Setup cell
# 1) Set reproducibility and print environment
import sys
import platform
from datetime import datetime
import os
from pathlib import Path
import json

SEED = 1234
import random
random.seed(SEED)
import numpy as np
np.random.seed(SEED)

import torch
torch.manual_seed(SEED)

print('Python:', sys.version)
print('Platform:', platform.platform())
print('Torch:', torch.__version__)
print('Numpy:', np.__version__)

GIT_HASH = None
try:
    import subprocess
    GIT_HASH = subprocess.check_output(['git','rev-parse','--short','HEAD']).decode().strip()
except Exception:
    GIT_HASH = 'N/A'
print('Git commit:', GIT_HASH)


In [ ]:
# 2) Create images folder & naming conventions
from pathlib import Path
IMAGES_DIR = Path('../q1_image_captioning/images').resolve()
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

from datetime import datetime

def make_fig_name(section, metric, desc, ext='png'):
    ts = datetime.now().strftime('%Y%m%d-%H%M%S')
    name = f"fig-{section}-{metric}-{desc}-{ts}.{ext}"
    name = name.replace(' ', '-').lower()
    return IMAGES_DIR / name

print('Images will be saved to:', IMAGES_DIR)
print('Example name:', make_fig_name('captioning','hist','length'))

In [ ]:
# 3) Install dependencies guard and imports
# Note: For CI we include optional installs. Uncomment if needed.
# !pip install --upgrade --quiet matplotlib seaborn pillow pandas

import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from PIL import Image, ImageDraw, ImageFont

print('matplotlib:', matplotlib.__version__)
print('seaborn:', sns.__version__)
print('pandas:', pd.__version__)
print('PIL:', Image.__version__)


In [ ]:
# 4) Load data (or create synthetic data) and diagnostic plot
from q1_image_captioning.tokenizer import Tokenizer

# Try to locate a CSV in data; otherwise, make synthetic captions
DATA_CSV = Path('../data/flickr_captions_subset.csv').resolve()
if DATA_CSV.exists():
    df = pd.read_csv(DATA_CSV)
    captions = df['caption'].tolist()
else:
    print('No real dataset found; generating synthetic captions for demo')
    captions = [
        'A cat sits on the window',
        'Two people are walking their dog',
        'A group of children play in a park',
        'An airplane flies over the mountains',
        'A red car parked next to a tree',
    ] * 50

# Build tokenizer and vocab
tokenizer = Tokenizer()
tokenizer.build_vocab(captions, min_freq=1)
print('Vocab size:', tokenizer.vocab_size())

# Caption lengths
lengths = [len(tokenizer._tokenize_text(c)) + 2 for c in captions]  # +2 for <s>/<e>

fig, ax = plt.subplots(figsize=(3.5, 2.5))
sns.histplot(lengths, bins=10, kde=False, color='C0')
ax.set_title('Caption length distribution')
ax.set_xlabel('Tokens')
ax.set_ylabel('Count')

from utils.utils import save_figure
hist_path = make_fig_name('captioning','length-hist','tokens')
saved = save_figure(fig, str(hist_path))
print('Saved histogram to:', saved)

# Add a small manifest structure
manifest = []
manifest.append({'filename': saved, 'width_in': 3.5, 'height_in': 2.5, 'dpi': 300, 'caption_placeholder': 'Caption length distribution'})



In [ ]:
# 5) Create synthetic image and save it, then render sample caption on image
sample_img = Image.new('RGB', (224,224), color=(73,109,137))
d = ImageDraw.Draw(sample_img)
d.text((10,10), 'Sample Image: Cat', fill=(255,255,255))
img_path = IMAGES_DIR / 'q1_sample_image_01_cat.png'
sample_img.save(img_path)
print('Saved sample image to', img_path)

# Save sample caption text
sample_caption = 'A cat sits on the window'
text_path = IMAGES_DIR / 'q1_sample_caption_01.txt'
with open(text_path, 'w') as f:
    f.write(sample_caption)
print('Saved sample caption to', text_path)

# Render caption text onto an image for the report
img_with_caption = Image.new('RGB', (800,200), (255,255,255))
d = ImageDraw.Draw(img_with_caption)
try:
    font = ImageFont.truetype('DejaVuSans.ttf', 18)
except Exception:
    font = ImageFont.load_default()
d.text((10,10), sample_caption, fill=(0,0,0), font=font)
caption_img_path = IMAGES_DIR / 'q1_generated_caption_sample_01.png'
save_figure(img_with_caption, str(caption_img_path))
manifest.append({'filename': str(caption_img_path), 'width_in': 8.0, 'height_in': 2.0, 'dpi': 300, 'caption_placeholder': 'Generated caption sample'})
print('Saved caption image to', caption_img_path)


In [ ]:
# 6) Save manifest to images/manifest.csv and export LaTeX-ready table
import csv
manifest_path = IMAGES_DIR / 'manifest.csv'
with open(manifest_path, 'w', newline='') as csvfile:
    writer = csv.DictWriter(csvfile, fieldnames=['filename','width_in','height_in','dpi','caption_placeholder'])
    writer.writeheader()
    for r in manifest:
        writer.writerow(r)

# Save a short CSV of sample captions for reproducibility
sample_table = pd.DataFrame({'image':[str(img_path.name)], 'caption':[sample_caption]})
sample_table.to_csv(IMAGES_DIR / 'table-sample-captions.csv', index=False)
sample_table.to_latex(IMAGES_DIR / 'table-sample-captions.tex', index=False)
print('Saved manifest and tables to', IMAGES_DIR)


In [ ]:
# 8) Small smoke training run (quick)
print('Starting small smoke training run (this may take a minute)')
from q1_image_captioning.train import train_smoke
train_smoke(device='cpu')
print('Smoke training done; check images folder for loss curve and sample predictions')

In [ ]:
# 7) Verification checks
from PIL import Image as PILImage
# Check files exist and have reasonable sizes
expected_files = [hist_path, img_path, caption_img_path, manifest_path]
for p in expected_files:
    p = Path(p)
    assert p.exists(), f"Expected file missing: {p}"
    if p.suffix.lower() in ['.png', '.jpg', '.jpeg']:
        w,h = PILImage.open(p).size
        print(p.name, 'size', w, 'x', h)
    else:
        print(p.name, 'size', p.stat().st_size)

print('All verification checks passed')


In [ ]:
# 9) Inference and BLEU evaluation
from q1_image_captioning.infer import run_inference
images_dir = '../q1_image_captioning/images'
ckpt_dir = images_dir + '/checkpoints'
result = run_inference(images_dir, ckpt_dir, n=5, device='cpu')
print('BLEU scores:', result['bleu'])
for ref, hyp in zip(result['references'], result['hypotheses']):
    print('GT:', ref)
    print('PR:', hyp)
    print('---')
import pandas as pd
df = pd.DataFrame({'reference': result['references'], 'hypothesis': result['hypotheses']})
df.to_csv(images_dir + '/table-inference-results.csv', index=False)
df.to_latex(images_dir + '/table-inference-results.tex', index=False)
print('Saved inference results table to images folder')